# Ring-Light Best Stack — Forehead Lab Inference (Colab)

Production-style **chart-free** forehead Lab from Variable Lighting **torch zips**. ROI matches **FitSkin forehead**.

## Stack (best deployment)

| Step | What |
|---|---|
| 1 | Pre-AWB demosaic → reflectance \(R_0=\sqrt{A_0\odot B_0'}\) |
| 2 | Apple Vision **forehead** mask |
| 3 | **`tier3_affine`** indoor RGB→XYZ |
| 4 | **`hybrid_deploy` CAT** — Lu+torch SPD on F12/warm; frozen 5500 K on D65 |
| 5 | **Illuminant-routed multi-Lab corrector** |
| 6 | FairFace-7 → **specular_tone** (+ cheek pool when forehead L* is uniform) → Lab |

## Run order

1. **Runtime → GPU** optional
2. **Cell 1** — clone repo, extract assets, mount Drive
3. **Cell 2b** — Parker WB sweep from **`data/ring_light/demo_zips/`** in git (no Drive) — or Drive fallback
4. **Cell 3** — run pipeline + compare FitSkin ΔE by `wb_cell` (illumination grid + segmentation viz)
5. **Cell 4** — optional frozen vs best-stack on one zip
6. **Cell 6** — pinned n=84 cohort tables (optional; no re-run)

> Drive needs `Variable Lighting Ring Light/…/Parker P2/{D65,F12}/*.zip`.


## 0 — Setup


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ══════════════════════════════════════════════════════════════════════════════
# Install deps (IPython magics — Colab). Safe if already installed.
%pip install -q rawpy opencv-python-headless numpy matplotlib gdown

import json, sys, zipfile, subprocess, textwrap
from pathlib import Path

try:
    import torch  # noqa: F401
except ImportError:
    %pip install -q torch torchvision

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

REPO_URL = "https://github.com/RooneyEmily/Fitskin.git"


def _sh(cmd: str) -> None:
    """Run a shell command (works in Colab and local)."""
    print("+", cmd)
    subprocess.run(cmd, shell=True, check=False)


if Path("Fitskin").is_dir():
    _sh("cd Fitskin && git pull --ff-only 2>/dev/null || true")
    REPO = Path("Fitskin").resolve()
elif (Path.cwd() / "pipeline" / "d65_fairface7_roi.py").is_file():
    REPO = Path.cwd().resolve()
else:
    _sh(f"git clone -q {REPO_URL}")
    REPO = Path("Fitskin").resolve()

# Mount Drive (needed for Parker WB-sweep zips in Cell 2b). Do NOT deep-scan MyDrive here
# (rglob over all of Drive can hang for minutes and look like the cell is stuck).
if IN_COLAB:
    if not Path("/content/drive/MyDrive").is_dir():
        print("Mounting Google Drive …")
        drive.mount("/content/drive")
    md = Path("/content/drive/MyDrive")
    if md.is_dir():
        print("Drive OK:", md)
        top = sorted(md.glob("Variable Lighting*"))[:8]
        if top:
            print("Top-level lighting folders (My Drive):")
            for h in top:
                print(" ", h)
        else:
            print(
                "NOTE: no 'Variable Lighting*' folder in My Drive root.\n"
                "  If the booth data is under **Shared with me**, Colab only sees it after you\n"
                "  **Add shortcut to Drive** (Drive web → right-click folder → Add shortcut → My Drive).\n"
                "  Or put flat zip folders in My Drive and set EXTRA_ZIP_DIRS in Cell 2b."
            )
        sd = Path("/content/drive/Shareddrives")
        if sd.is_dir() and any(sd.iterdir()):
            print("Shared drives mounted:", sd)
            for h in sorted(sd.glob("Variable Lighting*"))[:4]:
                print(" ", h)
    else:
        print("WARN: Drive not mounted — authorize when prompted, then re-run this cell.")

sys.path = [str(REPO)] + [p for p in sys.path if Path(p).resolve() != REPO]

MULTI_LAB = REPO / "calibration" / "multi_illuminant_lab_affine" / "multi_illuminant_lab_affine.json"
PIPE = REPO / "pipeline" / "d65_fairface7_roi.py"
SKIN_ROI = REPO / "pipeline" / "skin_roi.py"
ASSET_ZIP = REPO / "colab_assets" / "ringlight_best_stack.zip"

if ASSET_ZIP.is_file():
    print("Extracting colab_assets/ringlight_best_stack.zip …")
    with zipfile.ZipFile(ASSET_ZIP) as zf:
        zf.extractall(REPO)
elif (not MULTI_LAB.is_file()) or (not PIPE.is_file()):
    print("WARN: missing assets — push colab_assets/ringlight_best_stack.zip to the repo")

# Colab git tip may still import mediapipe via physio_skin_lab_monk in the eval script.
# Ensure chart-free forehead path uses inline Lab binning (no MediaPipe).
EVAL_PY = REPO / "scripts" / "evaluate_pansor20_chartfree_d65.py"
_TRIM_HELPERS = """
def _clip_skin_trim_q(q: float) -> float:
    if q <= 0.0:
        return 0.0
    return min(float(q), 0.45)


def _apply_channel_quantile_trim(
    sel: np.ndarray,
    channel: np.ndarray,
    trim_lo: float,
    trim_hi: float,
) -> np.ndarray:
    out = sel.copy()
    tlo = _clip_skin_trim_q(trim_lo)
    thi = _clip_skin_trim_q(trim_hi)
    if tlo > 0.0:
        out &= channel >= float(np.quantile(channel, tlo))
    if thi > 0.0:
        out &= channel <= float(np.quantile(channel, 1.0 - thi))
    return out
"""

if EVAL_PY.is_file():
    ev = EVAL_PY.read_text(encoding="utf-8")
    if "from physio_skin_lab_monk import" in ev or (
        "def _apply_channel_quantile_trim" not in ev and "apply_skin_lab_binning" in ev
    ):
        print("Patching scripts/evaluate_pansor20_chartfree_d65.py (drop mediapipe import) …")
        ev = ev.replace(
            "    from physio_skin_lab_monk import _apply_channel_quantile_trim\n", ""
        )
        if "def _clip_skin_trim_q" not in ev and "def apply_skin_lab_binning" in ev:
            ev = ev.replace(
                "def apply_skin_lab_binning(",
                _TRIM_HELPERS.strip() + "\n\n\ndef apply_skin_lab_binning(",
                1,
            )
        EVAL_PY.write_text(ev, encoding="utf-8")
    if "from physio_skin_lab_monk import" in EVAL_PY.read_text(encoding="utf-8"):
        raise RuntimeError(f"Failed to patch {EVAL_PY} — still imports physio_skin_lab_monk")
else:
    print("WARN: missing", EVAL_PY)

# Self-heal forehead∪cheek pool helper if git/assets still lack it.
_POOL_LINES = [
    "def apple_face_forehead_lab_pool_mask(",
    "    landmarks: dict,",
    "    dst_h: int,",
    "    dst_w: int,",
    "    *,",
    "    linear_rgb=None,",
    "):",
    '    """Forehead union cheek mask for specular_tone Lab pooling."""',
    "    forehead = apple_face_skin_roi_mask(",
    '        landmarks, dst_h, dst_w, roi="forehead", linear_rgb=linear_rgb',
    "    )",
    "    _, cheek = apple_face_cheek_masks(landmarks, dst_h, dst_w)",
    "    return (forehead | (cheek > 0)).astype(bool)",
    "",
]
_POOL_FN = "\n".join(_POOL_LINES)

src = SKIN_ROI.read_text(encoding="utf-8") if SKIN_ROI.is_file() else ""
if "def apple_face_forehead_lab_pool_mask" not in src:
    print("Patching pipeline/skin_roi.py with apple_face_forehead_lab_pool_mask …")
    if not SKIN_ROI.is_file():
        raise FileNotFoundError(f"Missing {SKIN_ROI}")
    SKIN_ROI.write_text(src.rstrip() + "\n\n" + _POOL_FN + "\n", encoding="utf-8")

CAL_DIR = REPO / "calibration" / "tier3_affine"
FAIRFACE_DIR = REPO / "calibration" / "fairface"
FAIRFACE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path("/content/ringlight_best_stack_runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_DIR = Path("/content/ringlight_uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

TORCH_DIR = Path("/content/drive/MyDrive/Torch_meas")  # optional

FF7 = FAIRFACE_DIR / "res34_fair_align_multi_7_20190809.pt"
if not FF7.is_file():
    print("Downloading FairFace-7 weights (~82 MB)…")
    _sh(f'gdown 11y0Wi3YQf21a_VcspUV4FwqzhMcfaVAB -O "{FF7}"')
assert FF7.is_file(), "FairFace weights missing — check gdown / network"

# Drop stale modules so patched files are loaded in later cells
for mod in list(sys.modules):
    if (
        mod == "pipeline"
        or mod.startswith("pipeline.")
        or mod == "scripts"
        or mod.startswith("scripts.")
    ):
        del sys.modules[mod]

import torch
from pipeline.d65_fairface7_roi import D65FairFace7ROIPipeline, write_result_json
from pipeline.skin_roi import apple_face_forehead_lab_pool_mask  # verify import

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("REPO:", REPO)
print("tier3 affine:", (CAL_DIR / "camera_rgb_to_xyz_affine.npy").is_file())
print("multi-lab corrector:", MULTI_LAB.is_file())
print("n=84 eval JSON:", (REPO / "data" / "ring_light" / "eval_n84_by_wb_cell.json").is_file())
print("forehead pool mask: OK")

# ── Torch flash SPD (used by hybrid_deploy Lu estimate) ───────────────────────
import matplotlib.pyplot as plt
import numpy as np
from pipeline.illuminant_estimation import load_torch_prior, load_torch_prior_from_cal_bundle

_torch_measured = False
try:
    _tp = load_torch_prior(TORCH_DIR)
    _torch_measured = True
    _torch_label = f"MK350 measured — {TORCH_DIR}"
except FileNotFoundError:
    _tp = load_torch_prior_from_cal_bundle(CAL_DIR)
    _torch_label = f"Calibration bundle fallback — {CAL_DIR.name}/iphone_calibration_bundle.json"

print(f"\nTorch flash prior: {_torch_label}")
print(f"  CCT ≈ {_tp.torch_cct_k:.0f} K  |  files={_tp.files}")

_spd = np.asarray(_tp.mean_spd, dtype=float)
_spd = _spd / max(float(np.nanmax(_spd)), 1e-8)
_fig, _ax = plt.subplots(figsize=(9, 3.2))
_ax.plot(_tp.wavelengths_nm, _spd, color="#d84a2b", lw=2.5)
_ax.set_xlabel("Wavelength (nm)")
_ax.set_ylabel("Normalized SPD")
_title = "iPhone torch SPD — used in Lu / hybrid_deploy CAT"
if not _torch_measured:
    _title += "\n(upload Torch_meas/ on Drive for measured ESPD; bundle fallback shown)"
_ax.set_title(_title, fontsize=10)
_ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("Setup OK.")


## Cells 2a / 2b — Get torch zips

**Cell 2b:** auto-finds **Parker** across all capture WB cells from Drive — **no upload**.

- Re-run **Cell 1** first and accept Drive.
- Needs `Parker P2/D65` and `Parker P2/F12` (or the full Variable Lighting tree) under My Drive.
- **Cell 2a** — leave skipped.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2a — OPTIONAL upload (skip — Cell 2b loads Parker from Drive)
# ══════════════════════════════════════════════════════════════════════════════
DO_UPLOAD = False
if DO_UPLOAD and IN_COLAB:
    from google.colab import files
    UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    for name in uploaded:
        (UPLOAD_DIR / name).write_bytes(uploaded[name])
        print("Saved", UPLOAD_DIR / name)
else:
    print("Upload skipped. Cell 2b finds Parker zips on Drive automatically.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2b — Parker × all capture WB cells (auto-find zips; no upload)
# ══════════════════════════════════════════════════════════════════════════════
#
# Filename cheat sheet (Parker-D65-A1Torch.zip):
#   • D65 / F12  = ring light program (booth illuminant)
#   • A–E        = capture wb_cell (phone white-balance setting; NOT applied in pipeline)
#   • 1,2,3…     = trial / replicate number (A1 = cell A, first take)
#
import re
from collections import defaultdict

WB_SWEEP_PERSON = "Parker"
MODE = "wb_sweep"
MANIFEST_PATH = REPO / "data" / "ring_light" / "wb_sweep_parker.json"
DEMO_DIR = REPO / "data" / "ring_light" / "demo_zips"
USE_REPO_ZIPS = True  # True = use zips shipped in git (no Drive / Shared-with-me)

# ── Optional: flat Drive download folders if repo zips missing ────────────────
# Colab example — Shared-with-me data must be shortcutted to My Drive, OR use flat folders:
# EXTRA_ZIP_DIRS = [
#     Path("/content/drive/MyDrive/drive-download-20260831T103650Z-1-001"),  # D65
#     Path("/content/drive/MyDrive/drive-download-20260831T103757Z-1-001"),  # F12
# ]
EXTRA_ZIP_DIRS = []  # e.g. local: Path.home() / "Downloads" / "drive-download-..."

_NAME_RE = re.compile(r"parker", re.I)

CAPTURE_WB_K = {
    "D65": {"A": 5500, "B": 6000, "C": 6500, "D": 7000, "E": 7500},
    "F12": {"A": 2500, "B": 2500, "C": 3000, "D": 3500, "E": 4000},
}
FITSKIN = {
    "D65": {"L": 55.99, "a": 11.89, "b": 20.53},
    "F12": {"L": 56.54, "a": 11.27, "b": 20.21},
}
NEEDED = [("D65", c) for c in "ABCDE"] + [("F12", c) for c in "BCDE"]

print(
    "Labeling: Parker-{D65|F12}-{A-E}{trial}Torch.zip  →  "
    "ring × capture wb_cell × replicate (pipeline ignores phone WB)"
)

if not MANIFEST_PATH.is_file():
    raise RuntimeError(f"Missing {MANIFEST_PATH} — re-run Cell 1 (asset zip includes wb_sweep_parker.json)")
man = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))


def _parse_parker_zip(stem: str):
    """Parse Parker-D65-A1Torch / Parker-F12C2Torch / Parker-D12-D1Torch ."""
    s = stem.replace(" ", "").strip()
    if not _NAME_RE.search(s):
        return None
    m = re.search(r"(D65|F12|D12|F1)[\-_]?([A-E])(\d+)", s, re.I)
    if not m:
        return None
    ill = m.group(1).upper()
    if ill in ("D12", "F1"):
        ill = "F12"
    return ill, m.group(2).upper(), int(m.group(3))


def _search_roots():
    """Booth tree, Shared drives, flat drive-download-* folders, user extras."""
    roots = []
    md = Path("/content/drive/MyDrive")
    shareddrives = Path("/content/drive/Shareddrives")

    def _add_variable_lighting_tree(base: Path) -> None:
        if not base.is_dir():
            return
        for p in [
            base / "Variable Lighting Ring Light" / "Variable Lighting Ring Light",
            base / "Variable Lighting Ring Light",
        ]:
            if p.is_dir():
                roots.append(p)
        for p in base.glob("Variable Lighting*"):
            if p.is_dir():
                roots.append(p)
                nested = p / "Variable Lighting Ring Light"
                if nested.is_dir():
                    roots.append(nested)
        for p in base.glob("drive-download-*"):
            if p.is_dir():
                roots.append(p)

    if md.is_dir():
        _add_variable_lighting_tree(md)
        # Shortcuts from Shared with me often sit one level deep in My Drive
        for p in md.glob("*Variable Lighting*"):
            if p.is_dir() and p not in roots:
                roots.append(p)
                _add_variable_lighting_tree(p)
        for base in list(roots):
            for person in base.glob("Parker*"):
                if person.is_dir():
                    roots.append(person)
                    for ill in ("D65", "F12"):
                        sub = person / ill
                        if sub.is_dir():
                            roots.append(sub)

    if shareddrives.is_dir():
        for team in shareddrives.iterdir():
            if team.is_dir():
                _add_variable_lighting_tree(team)
                for person in team.glob("**/Parker*"):
                    if person.is_dir():
                        roots.append(person)
                        for ill in ("D65", "F12"):
                            sub = person / ill
                            if sub.is_dir():
                                roots.append(sub)

    for base in [Path.home() / "Downloads", UPLOAD_DIR]:
        if base.is_dir():
            roots.append(base)
            for p in base.glob("drive-download-*"):
                if p.is_dir():
                    roots.append(p)

    roots.append(
        Path.home()
        / "Downloads"
        / "Variable Lighting Ring Light-20260829T185351Z-1-001"
        / "Variable Lighting Ring Light"
    )
    roots.append(REPO / "data" / "ring_light" / "demo_zips")

    for p in EXTRA_ZIP_DIRS:
        p = Path(p)
        if p.is_dir():
            roots.append(p)
        else:
            print("WARN: EXTRA_ZIP_DIRS path not found:", p)

    out, seen = [], set()
    for r in roots:
        r = Path(r)
        if not r.is_dir():
            continue
        key = str(r.resolve())
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out


def _glob_person_zips(root: Path):
    """Find Parker torch zips in root (flat folder) or one level down."""
    hits = []
    for pat in ("*Parker*Torch.zip", "*Parker*torch.zip", "*Parker*Torch .zip"):
        hits.extend(root.glob(pat))
    if not hits:
        for pat in ("*Parker*Torch.zip", "*Parker*torch.zip", "*Parker*Torch .zip"):
            hits.extend(root.rglob(pat))
    seen, out = set(), []
    for p in hits:
        k = str(p.resolve())
        if k not in seen:
            seen.add(k)
            out.append(p)
    return out


def _load_from_repo(manifest: dict):
    """Return (rows, zips) if all manifest files exist in DEMO_DIR."""
    if not USE_REPO_ZIPS or not DEMO_DIR.is_dir():
        return None
    rows, zips = [], []
    for d in manifest.get("demos", []):
        fname = d["file"]
        zp = DEMO_DIR / fname
        if not zp.is_file():
            for cand in DEMO_DIR.glob("*.zip"):
                if cand.name.replace(" ", "").lower() == fname.replace(" ", "").lower():
                    zp = cand
                    break
        if not zp.is_file():
            return None
        parsed = _parse_parker_zip(zp.stem)
        trial = parsed[2] if parsed else "?"
        rows.append(
            {
                "file": zp.name,
                "person": WB_SWEEP_PERSON,
                "illuminant": d["illuminant"],
                "wb_cell": d["wb_cell"],
                "trial": trial,
                "capture_wb_k": d.get("capture_wb_k") or CAPTURE_WB_K[d["illuminant"]][d["wb_cell"]],
                "fitskin_forehead": d.get("fitskin_forehead") or FITSKIN[d["illuminant"]],
            }
        )
        zips.append(zp)
    return rows, zips


_repo = _load_from_repo(man)
if _repo is not None:
    demo_rows, demo_zips = _repo
    candidates = list(demo_zips)
    demo_names_ordered = [r["file"] for r in demo_rows]
    print(f"\n✓ Loaded {len(demo_zips)} Parker zips from git ({DEMO_DIR.relative_to(REPO)}) — no Drive needed\n")
    print(f"{'idx':>3}  {'ring':3}  {'wb':2}  {'trial':5}  {'phone WB':>8}  file")
    print("-" * 72)
    for i, (meta, p) in enumerate(zip(demo_rows, demo_zips)):
        print(
            f"{i:3d}  {meta['illuminant']:3}  {meta['wb_cell']:2}  "
            f"{meta.get('trial', '?'):5}  {meta['capture_wb_k']!s:>7} K  {p.name}"
        )
    print(f"\nCell 3 will run all {len(demo_zips)} zips and compare ΔE by wb_cell.")
else:
    n_repo = len(list(DEMO_DIR.glob("Parker*.zip"))) if DEMO_DIR.is_dir() else 0
    print(f"\nRepo has {n_repo}/{len(man.get('demos', []))} Parker WB zips — searching Drive …")
    if IN_COLAB and not Path("/content/drive/MyDrive").is_dir():
        print("Mounting Google Drive …")
        from google.colab import drive
        drive.mount("/content/drive")
    print("\nSearching for Parker *Torch.zip …")
    roots = _search_roots()
    print(f"Search roots ({len(roots)}):")
    for r in roots:
        n = len(_glob_person_zips(r))
        tag = f"  ({n} Parker zips)" if n else ""
        print(f"  {r}{tag}")

    if not roots:
        raise RuntimeError(
            "No search roots. Mount Drive, set EXTRA_ZIP_DIRS, or push Parker zips to demo_zips on GitHub."
        )

    by_cell = defaultdict(list)
    scanned = 0
    unparsed = []
    for root in roots:
        for zp in _glob_person_zips(root):
            scanned += 1
            parsed = _parse_parker_zip(zp.stem)
            if not parsed:
                if _NAME_RE.search(zp.stem):
                    unparsed.append(zp.name)
                continue
            ill, cell, trial = parsed
            by_cell[(ill, cell)].append((trial, zp))

    print(
        f"\nParsed {sum(len(v) for v in by_cell.values())} Parker zips "
        f"→ {len(by_cell)} ring×wb_cell buckets (scanned {scanned} paths)"
    )
    if unparsed:
        print("Could not parse (check filename):", ", ".join(unparsed[:6]))

    preferred_name = {}
    for d in man.get("demos", []):
        preferred_name[(d["illuminant"], d["wb_cell"])] = d["file"]

    demo_rows, demo_zips, missing = [], [], []
    for ill, cell in NEEDED:
        cands = by_cell.get((ill, cell), [])
        if not cands:
            missing.append(f"{ill}-{cell}")
            wbk = CAPTURE_WB_K[ill][cell]
            print(
                f"WARN: no zip for ring {ill} wb_cell {cell} "
                f"(phone WB {wbk} K — look for Parker-{ill}-{cell}*Torch.zip)"
            )
            continue
        pref = preferred_name.get((ill, cell))
        chosen = None
        if pref:
            for trial, zp in cands:
                if zp.name == pref or zp.name.lower().replace(" ", "") == pref.lower():
                    chosen = zp
                    break
        if chosen is None:
            chosen = sorted(cands, key=lambda t: (t[0], t[1].name))[0][1]
        trial_n = next(t for t, zp in cands if zp == chosen)
        demo_rows.append(
            {
                "file": chosen.name,
                "person": WB_SWEEP_PERSON,
                "illuminant": ill,
                "wb_cell": cell,
                "trial": trial_n,
                "capture_wb_k": CAPTURE_WB_K[ill][cell],
                "fitskin_forehead": FITSKIN[ill],
            }
        )
        demo_zips.append(chosen)

    if not demo_zips:
        raise RuntimeError(
            "No Parker WB-sweep zips found.\n"
            "Push zips to GitHub demo_zips, set EXTRA_ZIP_DIRS, or add Shared-with-me shortcut to My Drive."
        )

    candidates = list(demo_zips)
    demo_names_ordered = [r["file"] for r in demo_rows]

    print(f"\nWB sweep person: {WB_SWEEP_PERSON}")
    print(
        f"Loaded {len(demo_zips)}/{len(NEEDED)} cells"
        + (f"  (missing: {', '.join(missing)})" if missing else "")
    )
    if missing:
        print(
            "\nTip: add flat download paths to EXTRA_ZIP_DIRS, or git push Parker zips to demo_zips/."
        )

    print(f"\n{'idx':>3}  {'ring':3}  {'wb':2}  {'trial':5}  {'phone WB':>8}  file")
    print("-" * 72)
    for i, (meta, p) in enumerate(zip(demo_rows, demo_zips)):
        print(
            f"{i:3d}  {meta['illuminant']:3}  {meta['wb_cell']:2}  "
            f"{meta.get('trial', '?'):5}  {meta['capture_wb_k']!s:>7} K  {p.name}"
        )
        print(f"      ↳ {p}")

    print(f"\nCell 3 will run all {len(demo_zips)} zips and compare ΔE by wb_cell.")


## 3 — Parker WB sweep inference

Runs every zip from Cell 2b, prints FitSkin ΔE **by capture wb_cell**, then:

- **`SHOW_ILLUM_GRID=True`** — face crops: D65 row vs F12 row
- **`SHOW_VIZ_INDEX=0`** — segmentation (forehead=green, cheek=blue, Lab pool=yellow)
- **`SHOW_ALL_SEGMENTATIONS=True`** — overlay montage for every capture


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Lihn WB sweep + ΔE comparison + illumination / segmentation viz
# ══════════════════════════════════════════════════════════════════════════════
import cv2
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from pathlib import Path
from delta_e_2000 import delta_e_2000
from scripts.evaluate_pansor20_chartfree_d65 import (
    extract_zip,
    linear_rgb_to_preview_bgr,
    load_apple_landmarks,
    load_dng_linear,
)
from pipeline.skin_roi import apple_face_cheek_masks, apple_face_skin_roi_mask

try:
    from pipeline.skin_roi import apple_face_forehead_lab_pool_mask
except ImportError:

    def apple_face_forehead_lab_pool_mask(landmarks, dst_h, dst_w, *, linear_rgb=None):
        forehead = apple_face_skin_roi_mask(
            landmarks, dst_h, dst_w, roi="forehead", linear_rgb=linear_rgb
        )
        _, cheek = apple_face_cheek_masks(landmarks, dst_h, dst_w)
        return (forehead | (cheek > 0)).astype(bool)

from models.fairface_race import face_rgb_crop_from_landmarks

CAT_MODE = "hybrid_deploy"
SAMPLING = "fairface7"
ROI = "forehead"
SHOW_VIZ_INDEX = 0  # deep-dive one zip (segmentation + Lab swatch)
SHOW_ILLUM_GRID = True  # thumbnail grid: D65 row + F12 row
SHOW_ALL_SEGMENTATIONS = False  # overlay every loaded zip (can be large)

if "from physio_skin_lab_monk import" in (REPO / "scripts" / "evaluate_pansor20_chartfree_d65.py").read_text(
    encoding="utf-8"
):
    raise RuntimeError("Re-run Cell 1 — eval script still imports mediapipe via physio_skin_lab_monk")

pipe = D65FairFace7ROIPipeline.from_defaults(
    cal_dir=CAL_DIR,
    fairface_dir=FAIRFACE_DIR,
    cat_mode=CAT_MODE,
    torch_dir=TORCH_DIR if TORCH_DIR.is_dir() else None,
    multi_lab_affine=MULTI_LAB,
    half_size=True,
    sampling=SAMPLING,
    roi=ROI,
)
print(f"Pipeline: roi={ROI}  cat_mode={CAT_MODE}  multi_lab={MULTI_LAB.name}  sampling={SAMPLING}")
if pipe.torch_prior is not None:
    tp = pipe.torch_prior
    _ts = "MK350 measured" if tp.n_files else "calibration bundle (Cell 1 plot)"
    print(f"Torch SPD in use: {_ts}  CCT≈{tp.torch_cct_k:.0f} K  files={tp.files}\n")
else:
    print()

cohort_results = {}
summary_rows = []
preview_cache = {}  # zip stem -> (preview_bgr, lm, result meta for viz)


def lab_to_srgb_u8(L, a, b):
    fy = (L + 16.0) / 116.0
    fx, fz = fy + a / 500.0, fy - b / 200.0
    eps, kappa = 216 / 24389, 24389 / 27

    def finv(t):
        return t**3 if t**3 > eps else (116 * t - 16) / kappa

    X, Y, Z = 0.95047 * finv(fx), finv(fy), 1.08883 * finv(fz)
    M = np.array(
        [[3.2406, -1.5372, -0.4986], [-0.9689, 1.8758, 0.0415], [0.0557, -0.2040, 1.0570]]
    )
    rgb = M @ np.array([X, Y, Z])
    lin2s = lambda u: 12.92 * u if u <= 0.0031308 else 1.055 * (max(u, 0) ** (1 / 2.4)) - 0.055
    return (np.clip([lin2s(float(c)) for c in rgb], 0, 1) * 255).astype(np.uint8)


def segmentation_overlay(preview_bgr, lm, linear_rgb, result):
    """Forehead=green, cheek=blue, Lab pool=yellow (forehead∪cheek when expanded)."""
    h, w = preview_bgr.shape[:2]
    forehead = apple_face_skin_roi_mask(lm, h, w, roi="forehead", linear_rgb=linear_rgb)
    _, cheek = apple_face_cheek_masks(lm, h, w)
    pool = apple_face_forehead_lab_pool_mask(lm, h, w, linear_rgb=linear_rgb)
    lab_mask = pool if result.get("lab_pool_expanded") else forehead

    ov = preview_bgr.copy().astype(np.float32)
    # cheek only (not forehead)
    cheek_only = (cheek > 0) & ~forehead
    ov[cheek_only] = 0.55 * ov[cheek_only] + 0.45 * np.array([60, 140, 255])  # blue
    ov[forehead] = 0.50 * ov[forehead] + 0.50 * np.array([40, 220, 80])  # green
    if result.get("lab_pool_expanded"):
        ov[lab_mask] = 0.45 * ov[lab_mask] + 0.55 * np.array([40, 255, 255])  # yellow pool
    return ov.astype(np.uint8), forehead, cheek, lab_mask


for i, zp in enumerate(demo_zips):
    meta = next((r for r in demo_rows if r.get("file") == zp.name), {})
    person = meta.get("person", "?")
    ill = meta.get("illuminant") or "?"
    wb = meta.get("wb_cell") or "?"
    wbk = meta.get("capture_wb_k")
    ref = meta.get("fitskin_forehead")
    try:
        r = pipe.run_zip(zp)
    except Exception as exc:
        print(f"[{i:02d}] FAIL {zp.name}: {exc}")
        summary_rows.append(
            {"idx": i, "file": zp.name, "person": person, "ill": ill, "wb": wb, "wbk": wbk, "error": str(exc)}
        )
        continue
    out_json = OUT_DIR / f"{zp.stem}.json"
    write_result_json(r, out_json)
    cohort_results[zp.name] = r
    pred = np.array([r["L"], r["a"], r["b"]], dtype=np.float64)
    de = float("nan")
    dL = da = db = float("nan")
    if ref:
        ref_lab = np.array([ref["L"], ref["a"], ref["b"]], dtype=np.float64)
        de = float(delta_e_2000(pred, ref_lab))
        dL, da, db = pred[0] - ref_lab[0], pred[1] - ref_lab[1], pred[2] - ref_lab[2]
    ef = r.get("exposure_flags") or {}
    summary_rows.append(
        {
            "idx": i,
            "file": zp.name,
            "person": person,
            "ill": ill,
            "wb": wb,
            "wbk": wbk,
            "L": r["L"],
            "a": r["a"],
            "b": r["b"],
            "de": de,
            "dL": dL,
            "da": da,
            "db": db,
            "cat_cct": r.get("cat_cct"),
            "l_sampling": r.get("l_sampling"),
            "pool": r.get("lab_pool_expanded"),
            "L_std": r.get("forehead_L_std"),
            "n_roi": r.get("n_roi"),
            "warn": "⚠" if ef.get("out_of_band") else "",
            "L_high": ef.get("L_ge_75"),
            "ref_L": ref["L"] if ref else None,
        }
    )

    # cache preview for illumination grid / segmentation
    try:
        work = OUT_DIR / "_viz_cache"
        nf, fl, lm_path = extract_zip(zp, work / zp.stem)
        A0 = load_dng_linear(nf, half_size=True, use_camera_wb=False)
        lm = load_apple_landmarks(lm_path)
        preview = linear_rgb_to_preview_bgr(A0)
        preview_cache[zp.stem] = (preview, lm, A0, r)
    except Exception as exc:
        print(f"  (viz cache skip {zp.name}: {exc})")

print(f"{'idx':>3}  {'ill':3}  {'wb':2}  {'ΔE':>5}  {'L*':>5}  {'ΔL*':>5}  {'pool':4}  file")
print("-" * 78)
for row in summary_rows:
    if row.get("error"):
        print(f"{row['idx']:3d}  FAIL  {row['file']}: {row['error'][:50]}")
        continue
    pool_s = "pool" if row.get("pool") else "fore"
    print(
        f"{row['idx']:3d}  {row['ill']:3}  {row['wb']:2}  {row['de']:5.2f}  "
        f"{row['L']:5.1f}  {row['dL']:+5.1f}  {pool_s:4}  {row['file']}{row.get('warn','')}"
    )

ok_de = [row["de"] for row in summary_rows if row.get("de") == row.get("de")]
if ok_de:
    print(f"\nMean ΔE₀₀ (n={len(ok_de)}): {sum(ok_de) / len(ok_de):.2f}")

# ── Why is D65 ΔE high? (diagnostic) ─────────────────────────────────────────
d65_rows = [r for r in summary_rows if r.get("ill") == "D65" and r.get("de") == r.get("de")]
f12_rows = [r for r in summary_rows if r.get("ill") == "F12" and r.get("de") == r.get("de")]
if d65_rows:
    mean_d65 = sum(r["de"] for r in d65_rows) / len(d65_rows)
    mean_L_d65 = sum(r["L"] for r in d65_rows) / len(d65_rows)
    ref_L = d65_rows[0].get("ref_L") or 55.24
    print("\n" + "=" * 72)
    print(f"D65 ΔE diagnostic ({WB_SWEEP_PERSON})")
    print("=" * 72)
    print(f"  FitSkin forehead ref L* ≈ {ref_L:.1f}  |  {WB_SWEEP_PERSON} D65 mean L* ≈ {mean_L_d65:.1f}  (ΔL* ≈ {mean_L_d65 - ref_L:+.1f})")
    print(f"  Mean ΔE₀₀ on D65: {mean_d65:.1f}  (cells range {min(r['de'] for r in d65_rows):.1f}–{max(r['de'] for r in d65_rows):.1f})")
    if f12_rows:
        mean_f12 = sum(r["de"] for r in f12_rows) / len(f12_rows)
        print(f"  Compare F12 mean ΔE₀₀: {mean_f12:.1f}  — warm ring + cheek pool usually tracks FitSkin")
    print(
        "  Likely cause: D65 captures run bright (L* 65–75). Pipeline CAT is OK (~5500 K frozen on D65);"
        "\n  error is mostly ΔL* (exposure), not wrong illuminant. Re-check ring SPD (~6500 K) vs capture exposure."
    )
    n_pool = sum(1 for r in d65_rows if r.get("pool"))
    print(f"  Cheek pool on D65: {n_pool}/{len(d65_rows)} zips (needs uniform forehead L* + Indian/F12 rule).")

cohort_summary = OUT_DIR / "cohort_summary.json"
cohort_summary.write_text(json.dumps({"n": len(summary_rows), "rows": summary_rows}, indent=2) + "\n")
print("\nWrote", cohort_summary)

# ── Compare ΔE across capture WB cells ───────────────────────────────────────
by_ill = defaultdict(list)
for row in summary_rows:
    if row.get("error") or row.get("de") != row.get("de"):
        continue
    by_ill[row["ill"]].append(row)

print("\n" + "=" * 72)
print(f"WB SWEEP — {WB_SWEEP_PERSON}  (pipeline ignores capture WB; pre-AWB + CAT)")
print("=" * 72)
for ill in ("D65", "F12"):
    rows_i = sorted(by_ill.get(ill, []), key=lambda r: r.get("wb") or "")
    if not rows_i:
        continue
    print(f"\n{ill} ring — FitSkin forehead ΔE₀₀ by capture wb_cell")
    print(f"{'wb':>3}  {'WB K':>5}  {'ΔE':>6}  {'L*':>6}  {'ΔL*':>6}  file")
    print("-" * 72)
    for r in rows_i:
        print(
            f"{r['wb']:>3}  {str(r.get('wbk', '?')):>5}  {r['de']:6.2f}  "
            f"{r['L']:6.1f}  {r['dL']:+6.1f}  {r['file']}"
        )
    des = [r["de"] for r in rows_i]
    best = min(rows_i, key=lambda r: r["de"])
    worst = max(rows_i, key=lambda r: r["de"])
    print(
        f"  mean={sum(des)/len(des):.2f}  "
        f"best={best['wb']}({best['de']:.2f})  worst={worst['wb']}({worst['de']:.2f})  "
        f"range={max(des)-min(des):.2f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, ill in zip(axes, ("D65", "F12")):
    rows_i = sorted(by_ill.get(ill, []), key=lambda r: r.get("wb") or "")
    if not rows_i:
        ax.set_visible(False)
        continue
    xs = [r["wb"] for r in rows_i]
    ys = [r["de"] for r in rows_i]
    ax.bar(xs, ys, color="#2563eb" if ill == "D65" else "#dc2626", alpha=0.85)
    for i, (r, y) in enumerate(zip(rows_i, ys)):
        ax.text(i, y + 0.15, f"{y:.1f}", ha="center", fontsize=8)
        if r.get("L_high") or r.get("warn"):
            ax.text(i, y * 0.5, "L*↑", ha="center", fontsize=7, color="white", fontweight="bold")
    ax.set_title(f"{ill} — {WB_SWEEP_PERSON}")
    ax.set_xlabel("capture wb_cell")
    ax.set_ylabel("ΔE₀₀ vs FitSkin forehead")
    ax.axhline(5, color="gray", ls=":", lw=0.8, alpha=0.6)
plt.suptitle("Capture WB factorial (phone WB not applied in pipeline)", fontsize=11)
plt.tight_layout()
plt.show()

# ── Illumination thumbnail grid (D65 vs F12 appearance) ────────────────────
if SHOW_ILLUM_GRID and preview_cache:
    cells_d65 = sorted(by_ill.get("D65", []), key=lambda r: r["wb"])
    cells_f12 = sorted(by_ill.get("F12", []), key=lambda r: r["wb"])
    ncols = max(len(cells_d65), len(cells_f12), 1)
    fig, axes = plt.subplots(2, ncols, figsize=(2.2 * ncols, 5.2))
    if ncols == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for j in range(ncols):
        ax = axes[0, j]
        if j < len(cells_d65):
            row = cells_d65[j]
            stem = Path(row["file"]).stem
            cached = preview_cache.get(stem)
            if cached:
                preview, lm, A0, res = cached
                face_rgb = face_rgb_crop_from_landmarks(preview, lm, padding=0.35)
                ax.imshow(face_rgb)
                ax.set_title(
                    f"D65 cell {row['wb']}  ΔE={row['de']:.1f}\nL*={row['L']:.0f} (ref {row.get('ref_L', 55):.0f})",
                    fontsize=8,
                )
            else:
                ax.text(0.5, 0.5, row["file"], ha="center", va="center", transform=ax.transAxes, fontsize=7)
        else:
            ax.axis("off")
        ax.set_ylabel("D65 ring" if j == 0 else "", fontsize=9)

    for j in range(ncols):
        ax = axes[1, j]
        if j < len(cells_f12):
            row = cells_f12[j]
            stem = Path(row["file"]).stem
            cached = preview_cache.get(stem)
            if cached:
                preview, lm, A0, res = cached
                face_rgb = face_rgb_crop_from_landmarks(preview, lm, padding=0.35)
                ax.imshow(face_rgb)
                ax.set_title(
                    f"F12 cell {row['wb']}  ΔE={row['de']:.1f}\nL*={row['L']:.0f} (ref {row.get('ref_L', 57):.0f})",
                    fontsize=8,
                )
            else:
                ax.text(0.5, 0.5, row["file"], ha="center", va="center", transform=ax.transAxes, fontsize=7)
        else:
            ax.axis("off")

    plt.suptitle(
        f"{WB_SWEEP_PERSON}: face crops under D65 (~6500 K ring) vs F12 (~3000 K ring)\n"
        "D65 often looks brighter / washed — high L* drives ΔE, not CAT failure",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

# ── Deep-dive: segmentation for one zip ──────────────────────────────────────
if SHOW_VIZ_INDEX is not None and summary_rows:
    vi = int(SHOW_VIZ_INDEX) % len(demo_zips)
    ZIP_PATH = demo_zips[vi]
    result = cohort_results.get(ZIP_PATH.name)
    meta = next((r for r in demo_rows if r.get("file") == ZIP_PATH.name), {})
    ref = meta.get("fitskin_forehead")
    if result is None:
        print(f"\nNo result for viz index {vi} ({ZIP_PATH.name})")
    else:
        print(f"\nSegmentation viz [{vi}] {ZIP_PATH.name}")
        cached = preview_cache.get(ZIP_PATH.stem)
        if cached is None:
            work = OUT_DIR / "_viz"
            nf, fl, lm_path = extract_zip(ZIP_PATH, work / ZIP_PATH.stem)
            A0 = load_dng_linear(nf, half_size=True, use_camera_wb=False)
            lm = load_apple_landmarks(lm_path)
            preview = linear_rgb_to_preview_bgr(A0)
        else:
            preview, lm, A0, _ = cached

        seg_ov, forehead, cheek, lab_mask = segmentation_overlay(preview, lm, A0, result)
        face_rgb = face_rgb_crop_from_landmarks(preview, lm, padding=0.35)
        pred_swatch = np.full((160, 160, 3), lab_to_srgb_u8(result["L"], result["a"], result["b"]), dtype=np.uint8)
        ref_swatch = None
        if ref:
            ref_swatch = np.full((160, 160, 3), lab_to_srgb_u8(ref["L"], ref["a"], ref["b"]), dtype=np.uint8)

        fig, ax = plt.subplots(2, 3, figsize=(12, 7.5))
        ax[0, 0].imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
        ax[0, 0].set_title("No-flash (pre-AWB preview)")
        ax[0, 0].axis("off")

        ax[0, 1].imshow(cv2.cvtColor(seg_ov, cv2.COLOR_BGR2RGB))
        pool_note = "yellow=Lab pool (forehead∪cheek)" if result.get("lab_pool_expanded") else "green=forehead Lab ROI"
        ax[0, 1].set_title(f"Segmentation\n{pool_note}\nblue=cheek band")
        ax[0, 1].axis("off")

        ax[0, 2].imshow(face_rgb)
        ax[0, 2].set_title(
            f"FairFace-7: {result.get('fairface_label')} → {result.get('predicted_ethnicity')}\n"
            f"sampling={result.get('l_sampling')}  n_roi={result.get('n_roi')}"
        )
        ax[0, 2].axis("off")

        ax[1, 0].imshow(pred_swatch)
        ax[1, 0].set_title(f"Predicted Lab\n({result['L']:.1f}, {result['a']:.1f}, {result['b']:.1f})")
        ax[1, 0].axis("off")

        if ref_swatch is not None:
            ax[1, 1].imshow(ref_swatch)
            de = float(delta_e_2000(
                np.array([result["L"], result["a"], result["b"]]),
                np.array([ref["L"], ref["a"], ref["b"]]),
            ))
            ax[1, 1].set_title(
                f"FitSkin ref ({meta.get('illuminant')})\n"
                f"({ref['L']:.1f}, {ref['a']:.1f}, {ref['b']:.1f})\nΔE₀₀={de:.2f}"
            )
        else:
            ax[1, 1].axis("off")

        ef = result.get("exposure_flags") or {}
        diag = (
            f"illuminant (zip): {result.get('illuminant_label')}\n"
            f"CAT CCT: {result.get('cat_cct')}  Lu: {result.get('lu_cct_k')}\n"
            f"forehead L* std: {result.get('forehead_L_std', 0):.2f}\n"
            f"cheek pool: {result.get('lab_pool_expanded')}\n"
            f"capture wb_cell: {meta.get('wb_cell')} ({meta.get('capture_wb_k')} K setting)\n"
            f"exposure: {'OUT OF BAND' if ef.get('out_of_band') else 'OK'}"
            + ("  L*≥75" if ef.get("L_ge_75") else "")
        )
        ax[1, 2].text(0.05, 0.95, diag, va="top", fontsize=10, family="monospace", transform=ax[1, 2].transAxes)
        ax[1, 2].axis("off")
        ax[1, 2].set_title("Diagnostics")

        plt.suptitle(f"{ZIP_PATH.name}  ·  {result.get('illuminant_label')} ring", fontsize=11)
        plt.tight_layout()
        plt.show()

# Optional: segmentation montage for every zip
if SHOW_ALL_SEGMENTATIONS and preview_cache:
    keys = [Path(r["file"]).stem for r in summary_rows if not r.get("error")]
    n = len(keys)
    ncols = min(5, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.5 * ncols, 2.8 * nrows))
    axes = np.atleast_2d(axes)
    for k, stem in enumerate(keys):
        i, j = divmod(k, ncols)
        ax = axes[i, j]
        row = next(r for r in summary_rows if Path(r["file"]).stem == stem)
        res = cohort_results.get(row["file"])
        preview, lm, A0, _ = preview_cache[stem]
        seg_ov, _, _, _ = segmentation_overlay(preview, lm, A0, res)
        ax.imshow(cv2.cvtColor(seg_ov, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{row['ill']} {row['wb']} ΔE={row['de']:.1f}", fontsize=8)
        ax.axis("off")
    for k in range(n, nrows * ncols):
        i, j = divmod(k, ncols)
        axes[i, j].axis("off")
    plt.suptitle("All captures — green forehead, blue cheek, yellow=Lab pool", fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    ZIP_PATH = demo_zips[0]
    result = cohort_results.get(ZIP_PATH.name)


## 4 — Optional: frozen vs best stack

Uses the zip selected by Cell 3 `SHOW_VIZ_INDEX` (or first loaded zip if viz skipped).


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Side-by-side: frozen 5500 K vs best stack (viz zip)
# ══════════════════════════════════════════════════════════════════════════════
if "ZIP_PATH" not in dir() or "result" not in dir() or result is None:
    raise RuntimeError("Run Cell 3 first")

pipe_frozen = D65FairFace7ROIPipeline.from_defaults(
    cal_dir=CAL_DIR,
    fairface_dir=FAIRFACE_DIR,
    cat_mode="frozen_5500",
    half_size=True,
    sampling=SAMPLING,
    roi=ROI,
)
r_frozen = pipe_frozen.run_zip(ZIP_PATH)

print(f"Zip: {ZIP_PATH.name}\n")
print(f"{'Arm':<22} {'L*':>7} {'a*':>7} {'b*':>7} {'CAT K':>8} {'corrector':<12}")
print("-" * 70)
for label, r in [("frozen_5500 (baseline)", r_frozen), ("best stack", result)]:
    print(
        f"{label:<22} {r['L']:7.2f} {r['a']:7.2f} {r['b']:7.2f} "
        f"{float(r.get('cat_cct') or 0):8.0f} {str(r.get('lab_corrector') or '-'):<12}"
    )
dL, da, db = result["L"] - r_frozen["L"], result["a"] - r_frozen["a"], result["b"] - r_frozen["b"]
print(f"\nΔLab (best − frozen): ΔL*={dL:+.2f}  Δa*={da:+.2f}  Δb*={db:+.2f}")

meta = next((r for r in demo_rows if r.get("file") == ZIP_PATH.name), {})
ref = meta.get("fitskin_forehead")
if ref:
    pred = np.array([result["L"], result["a"], result["b"]], dtype=np.float64)
    ref_lab = np.array([ref["L"], ref["a"], ref["b"]], dtype=np.float64)
    de_best = float(delta_e_2000(pred, ref_lab))
    de_frozen = float(delta_e_2000(np.array([r_frozen["L"], r_frozen["a"], r_frozen["b"]]), ref_lab))
    print(f"ΔE₀₀ vs FitSkin forehead: best={de_best:.2f}  frozen={de_frozen:.2f}")


## 5 — Optional extras

Cell 3 already runs the full Parker WB sweep. Set `RUN_BATCH=True` in the next cell only if Cell 2b found extra zips beyond the sweep.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — OPTIONAL batch extras
# ══════════════════════════════════════════════════════════════════════════════
RUN_BATCH = False

if RUN_BATCH and "candidates" in dir() and "demo_names_ordered" in dir():
    batch_zips = [p for p in candidates if p.name not in set(demo_names_ordered)]
    if not batch_zips:
        print("No extra zips beyond the Lihn WB sweep.")
    else:
        for i, zp in enumerate(batch_zips, 1):
            try:
                r = pipe.run_zip(zp)
                write_result_json(r, OUT_DIR / f"{zp.stem}.json")
                print(f"[{i}] {zp.name}  Lab=({r['L']:.1f},{r['a']:.1f},{r['b']:.1f})")
            except Exception as exc:
                print(f"[{i}] FAIL {zp.name}: {exc}")
else:
    print("Batch skipped (RUN_BATCH=False).")


## 6 — Full cohort reference (n=84): ΔE by wb_cell

Pre-computed locally — **does not re-run 84 trials**. Compare with Cell 3 (Parker, specular_tone deploy).

Loaded from repo, asset zip (Cell 1), or inline fallback if GitHub tip lags.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Pinned n=84 ΔE₀₀ by wb_cell (optional)
# ══════════════════════════════════════════════════════════════════════════════
import json
import matplotlib.pyplot as plt

EVAL_JSON = REPO / "data" / "ring_light" / "eval_n84_by_wb_cell.json"
if EVAL_JSON.is_file():
    payload = json.loads(EVAL_JSON.read_text(encoding="utf-8"))
    print(f"Loaded {EVAL_JSON.relative_to(REPO)}")
else:
    payload = json.loads(r"""{
  "n_trials": 84,
  "arms": [
    "frozen_5500",
    "hybrid_deploy",
    "hybrid_multi_lab"
  ],
  "capture_wb_k": {
    "D65": {
      "A": 5500,
      "B": 6000,
      "C": 6500,
      "D": 7000,
      "E": 7500
    },
    "F12": {
      "A": 2500,
      "B": 2500,
      "C": 3000,
      "D": 3500,
      "E": 4000
    }
  },
  "overall": {
    "frozen_5500": {
      "n": 84,
      "mean": 16.49,
      "median": 18.0993
    },
    "hybrid_deploy": {
      "n": 84,
      "mean": 15.4979,
      "median": 15.3681
    },
    "hybrid_multi_lab": {
      "n": 84,
      "mean": 11.9512,
      "median": 8.2112
    }
  },
  "by_wb_cell": {
    "A": {
      "frozen_5500": {
        "n": 7,
        "mean": 13.6845,
        "median": 9.5432
      },
      "hybrid_deploy": {
        "n": 7,
        "mean": 15.1011,
        "median": 14.9972
      },
      "hybrid_multi_lab": {
        "n": 7,
        "mean": 11.2031,
        "median": 5.5213
      }
    },
    "B": {
      "frozen_5500": {
        "n": 19,
        "mean": 16.3499,
        "median": 18.5838
      },
      "hybrid_deploy": {
        "n": 19,
        "mean": 15.0719,
        "median": 14.9664
      },
      "hybrid_multi_lab": {
        "n": 19,
        "mean": 11.6459,
        "median": 7.9201
      }
    },
    "C": {
      "frozen_5500": {
        "n": 15,
        "mean": 15.7455,
        "median": 17.8744
      },
      "hybrid_deploy": {
        "n": 15,
        "mean": 14.4533,
        "median": 15.2588
      },
      "hybrid_multi_lab": {
        "n": 15,
        "mean": 9.8142,
        "median": 7.9783
      }
    },
    "D": {
      "frozen_5500": {
        "n": 20,
        "mean": 16.744,
        "median": 18.0993
      },
      "hybrid_deploy": {
        "n": 20,
        "mean": 15.5134,
        "median": 15.0913
      },
      "hybrid_multi_lab": {
        "n": 20,
        "mean": 11.4626,
        "median": 8.1166
      }
    },
    "E": {
      "frozen_5500": {
        "n": 23,
        "mean": 17.7241,
        "median": 18.9857
      },
      "hybrid_deploy": {
        "n": 23,
        "mean": 16.6384,
        "median": 17.4443
      },
      "hybrid_multi_lab": {
        "n": 23,
        "mean": 14.2499,
        "median": 17.1817
      }
    }
  },
  "by_illuminant_wb_cell": {
    "D65": {
      "A": {
        "frozen_5500": {
          "n": 7,
          "mean": 13.6845,
          "median": 9.5432
        },
        "hybrid_deploy": {
          "n": 7,
          "mean": 15.1011,
          "median": 14.9972
        },
        "hybrid_multi_lab": {
          "n": 7,
          "mean": 11.2031,
          "median": 5.5213
        }
      },
      "B": {
        "frozen_5500": {
          "n": 9,
          "mean": 10.5864,
          "median": 9.2578
        },
        "hybrid_deploy": {
          "n": 9,
          "mean": 10.7605,
          "median": 9.2578
        },
        "hybrid_multi_lab": {
          "n": 9,
          "mean": 7.2228,
          "median": 3.9821
        }
      },
      "C": {
        "frozen_5500": {
          "n": 7,
          "mean": 10.6849,
          "median": 7.9344
        },
        "hybrid_deploy": {
          "n": 7,
          "mean": 10.6849,
          "median": 7.9344
        },
        "hybrid_multi_lab": {
          "n": 7,
          "mean": 7.2389,
          "median": 2.5117
        }
      },
      "D": {
        "frozen_5500": {
          "n": 10,
          "mean": 13.8798,
          "median": 9.1482
        },
        "hybrid_deploy": {
          "n": 10,
          "mean": 13.8798,
          "median": 9.1482
        },
        "hybrid_multi_lab": {
          "n": 10,
          "mean": 11.5226,
          "median": 5.1258
        }
      },
      "E": {
        "frozen_5500": {
          "n": 11,
          "mean": 13.6066,
          "median": 10.4018
        },
        "hybrid_deploy": {
          "n": 11,
          "mean": 13.6066,
          "median": 10.4018
        },
        "hybrid_multi_lab": {
          "n": 11,
          "mean": 11.5636,
          "median": 6.3295
        }
      }
    },
    "F12": {
      "B": {
        "frozen_5500": {
          "n": 10,
          "mean": 21.5371,
          "median": 21.9509
        },
        "hybrid_deploy": {
          "n": 10,
          "mean": 18.9521,
          "median": 19.4952
        },
        "hybrid_multi_lab": {
          "n": 10,
          "mean": 15.6267,
          "median": 17.1721
        }
      },
      "C": {
        "frozen_5500": {
          "n": 8,
          "mean": 20.1735,
          "median": 19.6533
        },
        "hybrid_deploy": {
          "n": 8,
          "mean": 17.7506,
          "median": 17.0461
        },
        "hybrid_multi_lab": {
          "n": 8,
          "mean": 12.0675,
          "median": 8.2455
        }
      },
      "D": {
        "frozen_5500": {
          "n": 10,
          "mean": 19.6081,
          "median": 18.7441
        },
        "hybrid_deploy": {
          "n": 10,
          "mean": 17.147,
          "median": 16.1013
        },
        "hybrid_multi_lab": {
          "n": 10,
          "mean": 11.4025,
          "median": 9.235
        }
      },
      "E": {
        "frozen_5500": {
          "n": 12,
          "mean": 21.4985,
          "median": 21.319
        },
        "hybrid_deploy": {
          "n": 12,
          "mean": 19.4175,
          "median": 19.1955
        },
        "hybrid_multi_lab": {
          "n": 12,
          "mean": 16.7123,
          "median": 17.1841
        }
      }
    }
  },
  "source_csv": "/home/mabl-main/color_space_research/new/Fitskin/results/torch_illuminant_ringlight/torch_illuminant_ringlight.csv",
  "source_note": "n=84 manifest_ring_cc_all; forehead ROI; trimmed-mean Lab (l_sampling=off); tier3 affine + hybrid_deploy + multi-lab; vs FitSkin forehead reference."
}""")
    EVAL_JSON.parent.mkdir(parents=True, exist_ok=True)
    EVAL_JSON.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    print("Using inline n=84 eval table (file not in git clone yet); wrote", EVAL_JSON)

ARMS = payload.get("arms") or ["frozen_5500", "hybrid_deploy", "hybrid_multi_lab"]
WB_K = payload.get("capture_wb_k") or {}
cells = ["A", "B", "C", "D", "E"]

print(payload.get("source_note", ""))
print(f"n={payload.get('n_trials')} trials\n")

ov = payload.get("overall", {}).get("hybrid_multi_lab", {})
if ov.get("mean") is not None:
    print(f"Overall hybrid+multi-lab mean ΔE₀₀: {ov['mean']:.2f}  (median {ov.get('median', float('nan')):.2f})\n")

for ill in ("D65", "F12"):
    grp = payload.get("by_illuminant_wb_cell", {}).get(ill, {})
    if not grp:
        continue
    print(f"=== {ill} — hybrid+multi-lab mean ΔE by wb_cell ===")
    for cell in cells:
        if cell not in grp:
            continue
        st = grp[cell].get("hybrid_multi_lab", {})
        if st.get("mean") is None:
            continue
        wbk = WB_K.get(ill, {}).get(cell, "?")
        print(f"  {cell} ({wbk} K): {st['mean']:.2f}  n={st['n']}")
    print()

best = "hybrid_multi_lab"
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, ill in zip(axes, ("D65", "F12")):
    grp = payload.get("by_illuminant_wb_cell", {}).get(ill, {})
    xs, ys = [], []
    for cell in cells:
        st = grp.get(cell, {}).get(best)
        if st and st.get("mean") is not None:
            xs.append(cell)
            ys.append(st["mean"])
    if xs:
        ax.bar(xs, ys, color="#2563eb" if ill == "D65" else "#dc2626", alpha=0.85)
        for i, y in enumerate(ys):
            ax.text(i, y + 0.15, f"{y:.1f}", ha="center", fontsize=8)
    ax.set_title(f"{ill} cohort (n=84, trimmed-mean eval)")
    ax.set_xlabel("wb_cell")
    ax.set_ylabel("mean ΔE₀₀")
plt.suptitle("Pinned reference — trimmed mean; Cell 3 uses specular_tone on Parker", fontsize=10)
plt.tight_layout()
plt.show()
